In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

# Load data
df = pd.read_csv('/Users/claranatalies/Documents/year 4/COMP 3520 DS/PROJECT/DATASET/base.csv')
target = df['fraud_bool']
features = df.drop('fraud_bool', axis=1)

# Column definitions based on BAF dataset
numerical_cols = [
    'income', 'name_email_similarity', 'prev_address_months_count',
    'current_address_months_count', 'customer_age', 'days_since_request',
    'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h',
    'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w',
    'credit_risk_score', 'bank_months_count', 'proposed_credit_limit',
    'session_length_in_minutes', 'device_distinct_emails_8w', 'device_fraud_count',
    'month'
]

categorical_cols = [
    'payment_type', 'employment_status', 'email_is_free', 'housing_status',
    'phone_home_valid', 'phone_mobile_valid', 'has_other_cards', 'foreign_request',
    'source', 'device_os', 'keep_alive_session'
]

# 1. Handle missing values (-1) in numerical columns
features[numerical_cols] = features[numerical_cols].replace(-1, np.nan)
medians = features[numerical_cols].median()
features[numerical_cols] = features[numerical_cols].fillna(medians)

# 2. One-hot encode categoricals
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cats = encoder.fit_transform(features[categorical_cols])
cat_feature_names = encoder.get_feature_names_out(categorical_cols)
df_cats = pd.DataFrame(encoded_cats, columns=cat_feature_names, index=features.index)

# Drop original categoricals and concatenate
features = features.drop(columns=categorical_cols)
features = pd.concat([features, df_cats], axis=1)

# Save final feature column order (important!)
feature_cols = features.columns.tolist()

# 3. Scale all features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# 4. Save preprocessors
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(encoder, 'encoder.pkl')
joblib.dump(medians, 'medians.pkl')
joblib.dump(feature_cols, 'feature_cols.pkl')

print(f"✅ Preprocessing complete. Feature vector dimension: {scaled_features.shape[1]}")
print("Update your PostgreSQL table's VECTOR dimension to this value if needed.")

✅ Preprocessing complete. Feature vector dimension: 58
Update your PostgreSQL table's VECTOR dimension to this value if needed.


In [ ]:
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values, Json
import joblib

# Load preprocessors
scaler = joblib.load('scaler.' \
'')
encoder = joblib.load('encoder.pkl')
medians = joblib.load('medians.pkl')
feature_cols = joblib.load('feature_cols.pkl')

# Load original data
df = pd.read_csv('/Users/claranatalies/Documents/year 4/COMP 3520 DS/PROJECT/DATASET/base.csv')
target = df['fraud_bool']
features = df.drop('fraud_bool', axis=1)

# Column definitions (same as preprocess.py)
numerical_cols = [
    'income', 'name_email_similarity', 'prev_address_months_count',
    'current_address_months_count', 'customer_age', 'days_since_request',
    'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h',
    'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w',
    'credit_risk_score', 'bank_months_count', 'proposed_credit_limit',
    'session_length_in_minutes', 'device_distinct_emails_8w', 'device_fraud_count',
    'month'
]

categorical_cols = [
    'payment_type', 'employment_status', 'email_is_free', 'housing_status',
    'phone_home_valid', 'phone_mobile_valid', 'has_other_cards', 'foreign_request',
    'source', 'device_os', 'keep_alive_session'
]

# Handle missing values using saved medians
features[numerical_cols] = features[numerical_cols].replace(-1, np.nan)
features[numerical_cols] = features[numerical_cols].fillna(medians)

# One-hot encode categoricals
encoded_cats = encoder.transform(features[categorical_cols])
cat_feature_names = encoder.get_feature_names_out(categorical_cols)
df_cats = pd.DataFrame(encoded_cats, columns=cat_feature_names, index=features.index)

# Drop original categoricals and concatenate
features = features.drop(columns=categorical_cols)
features = pd.concat([features, df_cats], axis=1)

# Ensure column order matches training
features = features[feature_cols]

# Scale - use .values to avoid feature name validation
scaled_features = scaler.transform(features.values)

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="postgres",
    user="postgres",
    password="mysecretpassword"
)
cur = conn.cursor()

# Prepare rows for insertion
rows = []
for idx in range(len(df)):
    vec = scaled_features[idx].tolist()
    metadata = df.iloc[idx].drop('fraud_bool').to_dict()
    rows.append((
        int(target.iloc[idx]),
        int(df.iloc[idx]['month']),
        vec,
        Json(metadata)
    ))

# Batch insert
print("⏳ Inserting data... (may take several minutes)")
execute_values(cur, """
    INSERT INTO applications (fraud_bool, month, feature_vector, metadata)
    VALUES %s
""", rows, page_size=1000)
conn.commit()
cur.close()
conn.close()
print("✅ Data insertion complete.")

/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


⏳ Inserting data... (may take several minutes)
✅ Data insertion complete.


In [48]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine  # optional, removes warning

conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="postgres",
    user="postgres",
    password="mysecretpassword"
)
query = "SELECT id, fraud_bool, month, metadata FROM applications LIMIT 20;"

df = pd.read_sql(query, conn)
conn.close()

# Expand the JSON metadata into separate columns
metadata_expanded = pd.json_normalize(df['metadata'])
df_full = pd.concat([df[['id', 'fraud_bool', 'month']].reset_index(drop=True), 
                     metadata_expanded.reset_index(drop=True)], axis=1)

print("=== Data with metadata expanded ===")
df_full.head()

=== Data with metadata expanded ===


/var/folders/8z/b64380y12kv332c9qsmc494w0000gq/T/ipykernel_50096/1575166368.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,fraud_bool,month,month,income,source,device_os,velocity_4w,velocity_6h,customer_age,payment_type,velocity_24h,zip_count_4w,email_is_free,housing_status,foreign_request,has_other_cards,phone_home_valid,bank_months_count,credit_risk_score,employment_status,days_since_request,device_fraud_count,keep_alive_session,phone_mobile_valid,bank_branch_count_8w,name_email_similarity,proposed_credit_limit,intended_balcon_amount,device_distinct_emails_8w,prev_address_months_count,session_length_in_minutes,current_address_months_count,date_of_birth_distinct_emails_4w
0,1,0,0,0,0.3,INTERNET,linux,6742.080561,13096.035018,40,AA,7850.955007,1059,1,BC,0,0,0,9,163,CB,0.006735,0,1,1,5,0.986506,1500.0,102.453711,1,-1,16.224843,25,5
1,2,0,0,0,0.8,INTERNET,other,5941.664859,9223.283431,20,AD,5745.251481,1658,1,BC,0,0,1,2,154,CA,0.010095,0,1,1,3,0.617426,1500.0,-0.849551,1,-1,3.363854,89,18
2,3,0,0,0,0.8,INTERNET,windows,5992.555113,4471.472149,40,AB,5471.988958,1095,1,BC,0,0,0,30,89,CA,0.012316,0,0,1,15,0.996707,200.0,-1.490386,1,9,22.730559,14,11
3,4,0,0,0,0.6,INTERNET,linux,5970.336831,14431.993621,30,AB,6755.344479,3483,1,BC,0,0,0,1,90,CA,0.006991,0,1,1,11,0.475100,200.0,-1.863101,1,11,15.215816,14,13
4,5,0,0,0,0.9,INTERNET,other,5940.734212,7601.511579,40,AA,5124.046930,2339,0,BC,0,0,1,26,91,CA,5.742626,0,0,1,1,0.842307,200.0,47.152498,1,-1,3.743048,29,6
